# Task 3: Coarse-to-fine ensemble

Add only the official 2026task3 data. Enable GPU and Internet for ImageNet initialization.
Training outputs are saved under /kaggle/working/task3. Collect the two final training outputs into one directory before running the fixed coordinate blends described in the README.

In [ ]:
%pip install -q timm pandas Pillow scikit-learn opencv-python-headless


In [ ]:
from pathlib import Path
Path("/kaggle/working/task3_source").mkdir(exist_ok=True)


In [ ]:
%%writefile /kaggle/working/task3_source/data.py
"""Official-data loading, retina geometry, pyramid rendering and label-synchronised augmentation."""
import math
import hashlib
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import pandas as pd
import cv2

cv2.setNumThreads(1)

FROZEN_FOLDS = dict(s.split(':') for s in (
    '0001:1,0002:1,0003:3,0004:0,0005:0,0006:3,0007:2,0008:1,0009:4,0010:3,0011:2,0012:1,0013:1,0014:1,0015:0,'
    '0016:4,0017:2,0018:0,0019:3,0020:4,0021:0,0022:4,0023:2,0024:4,0025:0,0026:2,0027:4,0028:0,0029:3,0030:0,'
    '0031:0,0032:4,0033:1,0034:2,0035:0,0036:2,0037:1,0038:3,0039:2,0040:0,0041:2,0042:1,0043:4,0044:2,0045:3,'
    '0046:0,0047:3,0048:1,0049:2,0050:4,0051:1,0052:2,0053:3,0054:2,0055:2,0056:1,0057:0,0058:0,0059:2,0060:4,'
    '0061:3,0062:3,0063:4,0064:4,0065:1,0066:4,0067:3,0068:3,0069:1,0070:4,0071:0,0072:4,0073:4,0074:3,0075:1,'
    '0076:2,0077:2,0078:3,0079:1,0080:3').split(','))

SPLIT2_FOLDS = dict(s.split(':') for s in (
    '0001:0,0002:4,0003:1,0004:2,0005:4,0006:1,0007:4,0008:4,0009:4,0010:1,0011:1,0012:3,0013:2,0014:3,0015:4,'
    '0016:2,0017:3,0018:4,0019:2,0020:2,0021:2,0022:1,0023:3,0024:0,0025:0,0026:4,0027:1,0028:3,0029:3,0030:1,'
    '0031:3,0032:3,0033:2,0034:1,0035:1,0036:0,0037:2,0038:2,0039:0,0040:1,0041:3,0042:1,0043:4,0044:4,0045:3,'
    '0046:2,0047:3,0048:0,0049:2,0050:0,0051:1,0052:4,0053:4,0054:3,0055:2,0056:1,0057:2,0058:0,0059:1,0060:0,'
    '0061:0,0062:2,0063:4,0064:4,0065:4,0066:0,0067:3,0068:3,0069:3,0070:1,0071:0,0072:1,0073:2,0074:0,0075:4,'
    '0076:0,0077:0,0078:2,0079:0,0080:3').split(','))

def find_data_dir(hint=None):
    roots = [Path(hint)] if hint else [Path('/kaggle/input')]
    for root in roots:
        hits = sorted(root.rglob('fovea_localization_train_GT.csv')) if root.exists() else []
        if hits:
            return hits[0].parent
    raise FileNotFoundError('Attach the official 2026task3 competition data')

def stem_key(name):
    s = Path(str(name)).stem
    return str(int(s)) if s.isdigit() else s

def retina_circle(rgb):
    """Centre and radius of the bright field of view (largest connected component), original pixels."""
    h, w = rgb.shape[:2]
    f = 512 / max(h, w)
    small = cv2.resize(rgb, (max(1, round(w * f)), max(1, round(h * f))), interpolation=cv2.INTER_AREA)
    n, _, stats, _ = cv2.connectedComponentsWithStats((small.max(2) > 20).astype(np.uint8), 8)
    x, y, bw, bh = stats[1 + np.argmax(stats[1:, 4]), :4] if n > 1 else (0, 0, small.shape[1], small.shape[0])
    sx, sy = w / small.shape[1], h / small.shape[0]
    return (x + bw / 2) * sx, (y + bh / 2) * sy, max(bw * sx, bh * sy) / 2

class Fundus:
    """One image kept as an anti-aliased pyramid (1/2, 1/4, 1/8) plus its field-of-view circle."""

    def __init__(self, name, rgb, xy=None, fold=-1):
        self.name, self.fold = name, fold
        self.group = hashlib.sha256(str(rgb.shape).encode() + rgb.tobytes()).hexdigest()
        self.h, self.w = rgb.shape[:2]
        self.xy = None if xy is None else np.asarray(xy, float)
        self.cx, self.cy, self.r = retina_circle(rgb)
        self.levels, self.to_level, src = {}, {}, rgb
        for f in (2, 4, 8):
            lw, lh = max(1, round(self.w / f)), max(1, round(self.h / f))
            src = cv2.resize(src, (lw, lh), interpolation=cv2.INTER_AREA)
            fx, fy = self.w / lw, self.h / lh
            self.levels[f] = src

            self.to_level[f] = np.array([[1 / fx, 0, .5 / fx - .5], [0, 1 / fy, .5 / fy - .5], [0, 0, 1.]])

def _read(path):
    return cv2.cvtColor(cv2.imread(str(path), cv2.IMREAD_COLOR), cv2.COLOR_BGR2RGB)

def load_dataset(data_dir, split=1):
    if split not in (1, 2):
        raise ValueError("Choose grouped split 1 or 2")
    folds = FROZEN_FOLDS if split == 1 else SPLIT2_FOLDS
    data_dir = Path(data_dir)
    labels = pd.read_csv(data_dir / 'fovea_localization_train_GT.csv', dtype={'data': str})
    sample = pd.read_csv(data_dir / 'sample_submission.csv')
    files = {stem_key(p.name): p for p in (data_dir / 'train').glob('*.jpg')}
    rows = [(files[stem_key(r.data)], (r.Fovea_X, r.Fovea_Y)) for r in labels.itertuples()]
    tests = sorted((data_dir / 'test').glob('*.jpg'))
    with ThreadPoolExecutor(8) as pool:
        train = list(pool.map(lambda r: Fundus(r[0].name, _read(r[0]), r[1], int(folds[r[0].stem])), rows))
        test = list(pool.map(lambda p: Fundus(p.name, _read(p)), tests))
    assert len(train) == 80 and len(test) == 20 and len(sample) == 40 and list(sample) == ['ImageID', 'value']
    assert all(np.isfinite(fd.xy).all() and 0 <= fd.xy[0] < fd.w and 0 <= fd.xy[1] < fd.h for fd in train)
    for fold in range(5):
        if not {fd.group for fd in train if fd.fold == fold}.isdisjoint(
                {fd.group for fd in train if fd.fold != fold}):
            raise ValueError("Identical training images cross validation folds")
    return train, test, sample

def render(fd, M, size):
    """Warp from the pyramid level just above the needed resolution; M maps original pixels to input pixels."""
    scale = math.sqrt(abs(np.linalg.det(M[:2, :2])))
    ok = [f for f in fd.levels if 1 / f >= .98 * scale]
    f = max(ok) if ok else min(fd.levels)
    A = M @ np.linalg.inv(fd.to_level[f])
    return cv2.warpAffine(fd.levels[f], A[:2], (size, size), flags=cv2.INTER_LINEAR,
                          borderMode=cv2.BORDER_CONSTANT, borderValue=(0, 0, 0))

def square(cx, cy, side, size):
    """Isotropic map of a side x side window centred on (cx, cy) to a size x size input."""
    s = size / side
    return np.array([[s, 0, (size - 1) / 2 - cx * s], [0, s, (size - 1) / 2 - cy * s], [0, 0, 1.]])

def coarse_base(fd, size, margin=.03, crop='fov'):
    """'fov': isotropic square around the field-of-view circle; 'image': whole image squeezed to size x size."""
    if crop == 'image':
        sx, sy = size / fd.w, size / fd.h
        return np.array([[sx, 0, .5 * sx - .5], [0, sy, .5 * sy - .5], [0, 0, 1.]])
    return square(fd.cx, fd.cy, 2 * fd.r * (1 + margin), size)

def hflip(size):
    return np.array([[-1, 0, size - 1], [0, 1, 0], [0, 0, 1.]])

def apply(M, xy):
    p = M @ np.array([xy[0], xy[1], 1.])
    return p[:2] / p[2]

def random_affine(size, rng, rot, scale, shift, flip=True):
    c = (size - 1) / 2
    s = math.exp(rng.uniform(-math.log(scale), math.log(scale)))
    A = np.vstack([cv2.getRotationMatrix2D((c, c), rng.uniform(-rot, rot), s), [0, 0, 1.]])
    A[:2, 2] += rng.uniform(-shift, shift, 2) * size
    return hflip(size) @ A if flip and rng.random() < .5 else A

def photometric(img, rng, p=1.):
    """Colour, contrast, gamma, saturation, blur and noise jitter inside the retina; labels untouched."""
    x = img.astype(np.float32) / 255
    if p <= 0:
        return x
    inside = (img.max(2, keepdims=True) > 10).astype(np.float32)
    x = x * rng.uniform(1 - .1 * p, 1 + .1 * p, 3).astype(np.float32)
    x = (x - .5) * rng.uniform(1 - .25 * p, 1 + .25 * p) + .5 + rng.uniform(-.1 * p, .1 * p)
    x = np.clip(x, 0, 1) ** rng.uniform(1 - .2 * p, 1 + .25 * p)
    g = x.mean(2, keepdims=True)
    x = g + (x - g) * rng.uniform(1 - .3 * p, 1 + .3 * p)
    if rng.random() < .3 * p:
        x = cv2.GaussianBlur(x, (0, 0), rng.uniform(.3, 1.2))
    if rng.random() < .3 * p:
        x = x + rng.normal(0, .02 * p, x.shape).astype(np.float32)
    return (np.clip(x, 0, 1) * inside).astype(np.float32)

def random_erase(img, rng, p, frac=.25):
    """Blank one random rectangle (any position, the fovea included) so context must also be used."""
    if p <= 0 or rng.random() >= p:
        return img
    s = img.shape[0]
    h, w = (rng.uniform(.08, frac, 2) * s).astype(int)
    y, x = rng.integers(0, s - h), rng.integers(0, s - w)
    img = img.copy()
    img[y:y + h, x:x + w] = img[img.max(2) > 10].mean(0) if (img.max(2) > 10).any() else 0
    return img

def coarse_sample(fd, size, rng, aug):
    M = coarse_base(fd, size, crop=aug.get('crop', 'fov'))
    if aug['geo']:
        M = random_affine(size, rng, aug['rot'], aug['scale'], aug['shift'], aug.get('flip', True)) @ M
    img = random_erase(render(fd, M, size), rng, aug.get('erase', 0.))
    return photometric(img, rng, aug['photo']), apply(M, fd.xy)

def fine_sample(fd, size, crop, rng, aug):
    """Window around the label plus a random offset wider than typical coarse errors (avoids a centre bias)."""
    jitter = aug['jitter_px'] if 'jitter_px' in aug else aug['jitter'] * crop
    wide = aug['wide_px'] if 'wide_px' in aug else aug['wide'] * crop
    for _ in range(50):
        d = rng.normal(0, jitter, 2) if rng.random() < aug.get('p_narrow', .7) else rng.uniform(-wide, wide, 2)
        M = square(fd.xy[0] + d[0], fd.xy[1] + d[1], crop, size)
        if aug['geo']:
            M = random_affine(size, rng, aug['rot'], aug['scale'], 0, aug.get('flip', True)) @ M
        t = apply(M, fd.xy)
        if (t > .08 * size).all() and (t < .92 * size).all():
            break
    return photometric(render(fd, M, size), rng, aug['photo']), t


def dataset_fingerprint(root):
    """Bind every pipeline's outputs to the same official image files and label/template CSVs."""
    root = Path(root)
    digest = hashlib.sha256()
    files = [root / 'fovea_localization_train_GT.csv', root / 'sample_submission.csv']
    files += sorted((root / 'train').glob('*.jpg')) + sorted((root / 'test').glob('*.jpg'))
    for path in files:
        digest.update(str(path.relative_to(root)).encode())
        digest.update(hashlib.sha256(path.read_bytes()).digest())
    return digest.hexdigest()


In [ ]:
%%writefile /kaggle/working/task3_source/models.py
"""Heatmap U-Net on an ImageNet-pretrained timm encoder, soft-argmax decoding and the training loss."""
import torch
from torch import nn
from torch.nn import functional as F

class ConvGN(nn.Sequential):
    def __init__(self, a, b):
        super().__init__(nn.Conv2d(a, b, 3, padding=1, bias=False), nn.GroupNorm(8, b), nn.ReLU(inplace=True))

class HeatNet(nn.Module):
    """Encoder features at strides 4-32 are fused top-down; one logit map at stride 4 is returned."""
    stride = 4

    def __init__(self, encoder='resnet34', pretrained=True, dec=(256, 128, 64), drop=0.):
        super().__init__()
        import timm
        self.enc = timm.create_model(encoder, pretrained=pretrained, features_only=True)
        red, chs = self.enc.feature_info.reduction(), self.enc.feature_info.channels()
        self.pick = [red.index(r) for r in (4, 8, 16, 32)]
        c = [chs[i] for i in self.pick]
        self.d3 = nn.Sequential(ConvGN(c[3] + c[2], dec[0]), ConvGN(dec[0], dec[0]))
        self.d2 = nn.Sequential(ConvGN(dec[0] + c[1], dec[1]), ConvGN(dec[1], dec[1]))
        self.d1 = nn.Sequential(ConvGN(dec[1] + c[0], dec[2]), ConvGN(dec[2], dec[2]))
        self.drop = nn.Dropout2d(drop) if drop > 0 else nn.Identity()
        self.head = nn.Conv2d(dec[2], 1, 1)
        cfg = getattr(self.enc, 'pretrained_cfg', {}) or {}
        self.register_buffer('mean', torch.tensor(cfg.get('mean', (.485, .456, .406))).view(1, 3, 1, 1))
        self.register_buffer('std', torch.tensor(cfg.get('std', (.229, .224, .225))).view(1, 3, 1, 1))

    def forward(self, x):
        """x: RGB in [0, 1], shape (B, 3, S, S); returns logits (B, S/4, S/4)."""
        f = self.enc((x - self.mean) / self.std)
        f1, f2, f3, f4 = (f[i] for i in self.pick)

        def up(a, b):
            return torch.cat([F.interpolate(a, size=b.shape[-2:], mode='bilinear', align_corners=False), b], 1)

        y = self.d3(up(f4, f3))
        y = self.d2(up(y, f2))
        y = self.d1(up(y, f1))
        return self.head(self.drop(y)).squeeze(1)

def grid(n, stride, device):
    """Input-pixel coordinate of each heatmap cell centre (OpenCV pixel-centre convention)."""
    return (torch.arange(n, device=device, dtype=torch.float32) + .5) * stride - .5

def soft_argmax(logits, stride=4, temperature=1., window=0, prior_sigma=0.):
    """Expected position under softmax(logits / T).

    window keeps only cells near the peak; prior_sigma (input pixels) adds a Gaussian log-prior centred on the
    input centre, i.e. on the coarse estimate that the fine crop was centred on."""
    B, H, W = logits.shape
    z = logits.float().flatten(1) / temperature
    if prior_sigma:
        c = (W * stride - 1) / 2
        d2 = (grid(W, stride, z.device).view(1, W) - c) ** 2 + (grid(H, stride, z.device).view(H, 1) - c) ** 2
        z = z - (d2 / (2 * prior_sigma ** 2)).flatten()[None]
    if window:
        peak = z.argmax(1)
        py, px = (peak // W).view(-1, 1, 1), (peak % W).view(-1, 1, 1)
        yy = torch.arange(H, device=z.device).view(1, H, 1)
        xx = torch.arange(W, device=z.device).view(1, 1, W)
        keep = ((yy - py).abs() <= window) & ((xx - px).abs() <= window)
        z = z.masked_fill(~keep.flatten(1), float('-inf'))
    p = z.softmax(1).view(B, H, W)
    return torch.stack([(p.sum(1) * grid(W, stride, z.device)).sum(1),
                        (p.sum(2) * grid(H, stride, z.device)).sum(1)], 1)

DECODERS = {'global': dict(), 'sharp': dict(temperature=.5), 'local': dict(window=6)}

def heat_loss(logits, xy, sigma, coord_w=1., stride=4):
    """Cross-entropy to a normalised Gaussian target plus L1 of the soft-argmax (in heatmap cells)."""
    B, H, W = logits.shape
    gx, gy = grid(W, stride, logits.device), grid(H, stride, logits.device)
    d2 = (gx.view(1, 1, W) - xy[:, 0].view(-1, 1, 1)) ** 2 + (gy.view(1, H, 1) - xy[:, 1].view(-1, 1, 1)) ** 2
    t = torch.exp(-d2 / (2 * sigma ** 2)).flatten(1)
    t = t / t.sum(1, keepdim=True).clamp_min(1e-12)
    ce = -(t * logits.float().flatten(1).log_softmax(1)).sum(1).mean()
    l1 = (soft_argmax(logits, stride) - xy).abs().sum(1).mean() / stride
    return ce + coord_w * l1, ce.detach(), l1.detach()


In [ ]:
%%writefile /kaggle/working/task3_source/training.py
"""Training loop (parameter groups, warmup-cosine, EMA weights) and affine-TTA prediction."""
import copy
import math
import random
import time
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import torch
from data import render, apply, hflip
from models import HeatNet, heat_loss, soft_argmax, DECODERS

def default_device():
    if torch.cuda.is_available():
        return torch.device('cuda')
    if torch.backends.mps.is_available():
        return torch.device('mps')
    return torch.device('cpu')

def seed_all(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

def batch_stream(make, items, batch, steps, seed, workers=6):
    """Shuffled passes over the training images; samples are built by threads one batch ahead."""
    rng = np.random.default_rng(seed)
    reps = math.ceil(steps * batch / len(items)) + 1
    order = np.concatenate([rng.permutation(len(items)) for _ in range(reps)])[:steps * batch]
    seeds = rng.integers(0, 2 ** 31 - 1, len(order))
    with ThreadPoolExecutor(workers) as pool:
        futures = {}

        def submit(i):
            if i < len(order):
                futures[i] = pool.submit(make, items[order[i]], np.random.default_rng(seeds[i]))

        for i in range(3 * batch):
            submit(i)
        for s in range(steps):
            out = [futures.pop(s * batch + j).result() for j in range(batch)]
            for i in range((s + 3) * batch, (s + 4) * batch):
                submit(i)
            x = torch.from_numpy(np.ascontiguousarray(np.stack([o[0] for o in out]).transpose(0, 3, 1, 2)))
            y = torch.from_numpy(np.stack([o[1] for o in out]).astype(np.float32))
            yield x, y

def build_model(cfg, pretrained=True):
    return HeatNet(cfg['encoder'], pretrained, drop=cfg.get('drop', 0.))

def train_model(cfg, items, make, device, seed, log=print, val_fn=None, val_every=100):
    """val_fn(ema_model) -> dict is only logged (learning curves); it never selects weights or steps."""
    seed_all(seed)
    model = build_model(cfg, cfg.get('pretrained', True)).to(device)
    cuda = device.type == 'cuda'
    if cuda:
        model = model.to(memory_format=torch.channels_last)
    enc = [p for n, p in model.named_parameters() if n.startswith('enc.')]
    dec = [p for n, p in model.named_parameters() if not n.startswith('enc.')]
    opt = torch.optim.AdamW([{'params': enc, 'lr': cfg['lr_enc']}, {'params': dec, 'lr': cfg['lr_dec']}],
                            weight_decay=cfg.get('wd', 1e-2))
    steps = cfg['steps']
    warm = max(1, int(.05 * steps))
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: (s + 1) / warm if s < warm else
                                              .02 + .98 * .5 * (1 + math.cos(math.pi * (s - warm) / max(1, steps - warm))))
    scaler = torch.amp.GradScaler('cuda', enabled=cuda)
    ema = copy.deepcopy(model)
    ema_f = [v for v in ema.state_dict().values() if v.dtype.is_floating_point]
    cur_f = [v for v in model.state_dict().values() if v.dtype.is_floating_point]
    model.train()
    if cfg.get('freeze_bn'):

        for m in model.enc.modules():
            if isinstance(m, torch.nn.modules.batchnorm._BatchNorm):
                m.eval()
    t0, hist = time.time(), []
    for step, (x, y) in enumerate(batch_stream(make, items, cfg['batch'], steps, seed, cfg.get('workers', 6))):
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        if cuda:
            x = x.contiguous(memory_format=torch.channels_last)
        with torch.autocast(device.type, dtype=torch.float16, enabled=cuda):
            logits = model(x)
        loss, ce, l1 = heat_loss(logits.float(), y, cfg['sigma'], cfg.get('coord_w', 1.))
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.)
        scaler.step(opt)
        scaler.update()
        sched.step()
        decay = min(cfg.get('ema', .995), (1 + step) / (10 + step))
        with torch.no_grad():
            torch._foreach_mul_(ema_f, decay)
            torch._foreach_add_(ema_f, cur_f, alpha=1 - decay)
        if (step + 1) % val_every == 0 or step == 0:
            rec = dict(step=step + 1, loss=float(loss.detach()), ce=float(ce), l1_cells=float(l1),
                       lr_enc=opt.param_groups[0]['lr'], seconds=time.time() - t0)
            if val_fn is not None and (step + 1) % val_every == 0:
                _sync_buffers(model, ema)
                rec.update(val_fn(ema.eval()))
                ema.train()
            hist.append(rec)
            log(rec)
    _sync_buffers(model, ema)
    return ema.eval(), hist

def _sync_buffers(model, ema):
    for (k, v) in model.state_dict().items():
        if not v.dtype.is_floating_point:
            ema.state_dict()[k].copy_(v)

@torch.no_grad()
def predict(model, items, base_fn, size, device, flips=(False, True), batch=8, decoders=None):
    """Original-pixel predictions for every decoder, without ('plain') and with ('flip') mirror TTA.

    base_fn(fd) returns the 3x3 map from original pixels to network input; TTA views are inverted exactly."""
    model.eval()
    cuda = device.type == 'cuda'
    decoders = decoders or DECODERS
    out = {(d, f): [] for d in decoders for f in flips}
    for start in range(0, len(items), batch):
        chunk = items[start:start + batch]
        for flip in flips:
            Ms = [hflip(size) @ base_fn(fd) if flip else base_fn(fd) for fd in chunk]
            x = np.stack([render(fd, M, size) for fd, M in zip(chunk, Ms)]).astype(np.float32) / 255
            x = torch.from_numpy(np.ascontiguousarray(x.transpose(0, 3, 1, 2))).to(device)
            if cuda:
                x = x.contiguous(memory_format=torch.channels_last)
            with torch.autocast(device.type, dtype=torch.float16, enabled=cuda):
                logits = model(x).float()
            for d, kw in decoders.items():
                p = soft_argmax(logits, model.stride, **kw).cpu().numpy()
                out[(d, flip)].append(np.stack([apply(np.linalg.inv(M), q) for M, q in zip(Ms, p)]))
    res = {}
    for d in decoders:
        plain = np.concatenate(out[(d, False)])
        res[(d, 'plain')] = plain
        if True in flips:
            res[(d, 'flip')] = (plain + np.concatenate(out[(d, True)])) / 2
    return res


In [ ]:
%%writefile /kaggle/working/task3_source/submission.py
"""Official 20-image, 40-row coordinate submission format."""
import re
import numpy as np
from data import stem_key


def write_submission(test, pred, sample, path):
    pred = np.asarray(pred)
    if len(test) != 20 or pred.shape != (20, 2) or not np.isfinite(pred).all():
        raise ValueError('Expected twenty finite (x, y) predictions')
    mapping = {stem_key(fd.name): point for fd, point in zip(test, pred)}
    if len(mapping) != 20 or list(sample) != ['ImageID', 'value'] or len(sample) != 40:
        raise ValueError('Invalid official image names or submission template')
    for fd, point in zip(test, pred):
        if not (0 <= point[0] < fd.w and 0 <= point[1] < fd.h):
            raise ValueError(f'Coordinate outside the original image: {fd.name}')
    values, seen = [], set()
    for image_id in sample.ImageID:
        match = re.fullmatch(r'(.+)_Fovea_([XY])', str(image_id))
        if match is None:
            raise ValueError('Invalid coordinate ID in submission template')
        name, axis = stem_key(match[1]), match[2]
        if name not in mapping or (name, axis) in seen:
            raise ValueError('Unknown or repeated coordinate ID')
        seen.add((name, axis))
        values.append(float(mapping[name][0 if axis == 'X' else 1]))
    if len(seen) != 40:
        raise ValueError('Incomplete coordinate submission')
    frame = sample.copy()
    frame['value'] = values
    frame.to_csv(path, index=False)
    return frame


In [ ]:
%%writefile /kaggle/working/task3_source/stages.py
"""Grouped five-fold coarse and fine stages, OOF summaries and submission export."""
import json
import time
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from data import load_dataset, find_data_dir, coarse_sample, fine_sample, coarse_base, square, stem_key
from training import train_model, predict, default_device
from submission import write_submission


def rmse(a, b):
    return float(np.sqrt(np.mean((np.asarray(a, float) - np.asarray(b, float)) ** 2)))

def metrics(target, pred, folds):
    e = np.linalg.norm(target - pred, axis=1)
    row = dict(rmse=rmse(target, pred), mean_ed=float(e.mean()), median_ed=float(np.median(e)),
               p90_ed=float(np.quantile(e, .9)), max_ed=float(e.max()))
    for k in range(5):
        row[f'fold{k}'] = rmse(target[folds == k], pred[folds == k])
    return row

def summarize(name, train, out):
    idx = {fd.name: i for i, fd in enumerate(train)}
    target = np.array([fd.xy for fd in train])
    folds = np.array([fd.fold for fd in train])
    parts = [np.load(out / f'fold{k}.npz') for k in range(5)]
    keys = sorted({k[4:] for k in parts[0].files if k.startswith('val|')})
    oof, test, rows = {}, {}, []
    for key in keys:
        pred = np.full((len(train), 2), np.nan)
        for z in parts:
            for n, p in zip(z['val_names'], z['val|' + key]):
                pred[idx[str(n)]] = p
        assert np.isfinite(pred).all()
        oof[key], test[key] = pred, np.mean([z['test|' + key] for z in parts], 0)
        rows.append(dict(model=name, variant=key, **metrics(target, pred, folds)))
    np.savez(out / 'oof.npz', **oof)
    np.savez(out / 'test.npz', **test)
    table = pd.DataFrame(rows).sort_values('rmse')
    table.to_csv(out / 'summary.csv', index=False)
    return table

def _fold_loop(name, cfg, train, test, out, device, folds, make, infer):
    out.mkdir(parents=True, exist_ok=True)
    identity = {'config': cfg,
                'train': [(fd.name, fd.fold, fd.group, fd.xy.tolist(), fd.w, fd.h) for fd in train],
                'test': [(fd.name, fd.group, fd.w, fd.h) for fd in test]}
    record = out / 'identity.json'
    normalized = json.loads(json.dumps(identity))
    if record.exists() and json.loads(record.read_text()) != normalized:
        raise ValueError('Input images, grouped split or configuration changed; use a new output directory')
    if not record.exists() and list(out.glob('fold*.npz')):
        raise ValueError('Existing predictions have no input identity record; use a new output directory')
    record.write_text(json.dumps(identity, indent=2))
    (out / 'config.json').write_text(json.dumps(cfg, indent=2))
    for k in folds:
        dest = out / f'fold{k}.npz'
        if dest.exists():
            with np.load(dest, allow_pickle=False) as saved:
                expected = [fd.name for fd in train if fd.fold == k]
                if saved['val_names'].astype(str).tolist() != expected:
                    raise ValueError('Cached validation images differ from the grouped split')
                for key in saved.files:
                    if key.startswith(('val|', 'test|')) and not np.isfinite(saved[key]).all():
                        raise ValueError('Nonfinite cached predictions')
            continue
        tr = [fd for fd in train if fd.fold != k]
        va = [fd for fd in train if fd.fold == k]
        t0 = time.time()
        target = np.array([fd.xy for fd in va])

        def val_fn(model):
            return {f'val_{key}': rmse(target, v) for key, v in infer(model, va).items()
                    if key.split('|')[0] in ('global', 'sharp', 'p25', 'sp25')}
        model, hist = train_model(cfg, tr, make, torch.device(device), 2026 + k + cfg.get('seed_offset', 0),
                                  log=lambda r: print(name, k, {a: round(b, 4) for a, b in r.items()}, flush=True),
                                  val_fn=val_fn if cfg.get('val_every') else None, val_every=cfg.get('val_every') or 100)
        result = {f'{split}|{key}': v for split, items in (('val', va), ('test', test))
                  for key, v in infer(model, items).items()}
        pd.DataFrame(hist).to_csv(out / f'history_fold{k}.csv', index=False)
        if cfg.get('save_weights'):
            torch.save({n: v.half() if v.is_floating_point() else v for n, v in model.state_dict().items()},
                       out / f'model_fold{k}.pt')
        np.savez(dest, val_names=np.array([fd.name for fd in va]), seconds=time.time() - t0, **result)
        print(name, 'fold', k, f'{time.time() - t0:.0f}s', flush=True)
        del model
    return summarize(name, train, out) if all((out / f'fold{k}.npz').exists() for k in range(5)) else None

def run_coarse(name, cfg, train, test, root, device, folds=range(5)):
    size = cfg['size']

    def make(fd, rng):
        return coarse_sample(fd, size, rng, cfg['aug'])

    def infer(model, items):
        res = predict(model, items, lambda fd: coarse_base(fd, size, crop=cfg['aug'].get('crop', 'fov')), size,
                      torch.device(device))
        return {f'{d}|{m}': v for (d, m), v in res.items()}
    return _fold_loop(name, cfg, train, test, Path(root) / name, device, folds, make, infer)

PRIORS = (10, 15, 20, 25, 35, 50)

def fine_decoders(cfg):
    """Gaussian prior around the crop centre (the coarse estimate); sigma given in original pixels."""
    k = cfg['size'] / cfg['crop']
    dec = {'global': dict(), 'sharp': dict(temperature=.5)}
    for p in PRIORS:
        dec[f'p{p}'] = dict(prior_sigma=p * k)
        dec[f'sp{p}'] = dict(prior_sigma=p * k, temperature=.5)
    return dec

def run_fine(name, cfg, coarse, coarse_key, train, test, root, device, folds=range(5)):
    """Crops are centred on the coarse OOF prediction (validation) or the coarse fold-mean (test)."""
    size, crop = cfg['size'], cfg['crop']
    centers = dict(zip([fd.name for fd in train], np.load(Path(root) / coarse / 'oof.npz')[coarse_key]))
    centers.update(zip([fd.name for fd in test], np.load(Path(root) / coarse / 'test.npz')[coarse_key]))

    def make(fd, rng):
        return fine_sample(fd, size, crop, rng, cfg['aug'])

    decoders = fine_decoders(cfg)

    def infer(model, items):
        cur, result = {fd.name: centers[fd.name] for fd in items}, {}
        for it in range(cfg['iters']):
            res = predict(model, items, lambda fd, c=dict(cur): square(*c[fd.name], crop, size), size,
                          torch.device(device), decoders=decoders)
            result.update({f'{d}|{m}|it{it + 1}': v for (d, m), v in res.items()})
            cur = dict(zip([fd.name for fd in items], res[(cfg.get('center_decoder', 'p25'), 'flip')]))
        return result
    cfg = dict(cfg, coarse=coarse, coarse_key=coarse_key)
    return _fold_loop(name, cfg, train, test, Path(root) / name, device, folds, make, infer)


In [ ]:
%%writefile /kaggle/working/task3_source/inference.py
"""Re-run inference of saved fold models with other decoders / TTA; nothing is retrained, OOF stays out-of-fold."""
import json
from pathlib import Path
import numpy as np
import torch
from data import square, coarse_base
from training import build_model, predict

def load_fold_model(run_dir, k, device):
    cfg = json.loads((Path(run_dir) / 'config.json').read_text())
    model = build_model(cfg, pretrained=False)
    state = torch.load(Path(run_dir) / f'model_fold{k}.pt', map_location='cpu')
    model.load_state_dict({n: v.float() if v.is_floating_point() else v for n, v in state.items()})
    return model.to(device).eval(), cfg

def _mean(dicts):
    return {k: np.mean([d[k] for d in dicts], 0) for k in dicts[0]}

def fine_views(model, items, centers, crop, size, device, decoders, scales=(1.,), shifts=((0., 0.),)):
    """Average over crop scales and centre shifts (fractions of the crop) of flip-TTA predictions."""
    views = []
    for s in scales:
        for dx, dy in shifts:
            base = (lambda fd, s=s, dx=dx, dy=dy:
                    square(centers[fd.name][0] + dx * crop, centers[fd.name][1] + dy * crop, crop * s, size))
            views.append(predict(model, items, base, size, device, decoders=decoders))
    return _mean(views)

def fine_infer(model, items, centers, cfg, device, decoders, iters=2, center_key=('global', 'flip'), **tta):
    out, cur = {}, dict(centers)
    for it in range(iters):
        res = fine_views(model, items, cur, cfg['crop'], cfg['size'], device, decoders, **tta)
        out.update({(d, m, f'it{it + 1}'): v for (d, m), v in res.items()})
        cur = dict(zip([fd.name for fd in items], res[center_key]))
    return out

def coarse_infer(model, items, cfg, device, decoders, scales=(1.,)):
    crop = cfg['aug'].get('crop', 'fov')
    views = [predict(model, items, lambda fd, s=s: coarse_base(fd, cfg['size'], margin=.03 + (s - 1), crop=crop),
                     cfg['size'], device, decoders=decoders) for s in scales]
    return _mean(views)


In [ ]:
%%writefile /kaggle/working/task3_source/cascade.py
"""Final pipeline: equal-weight coarse ensemble -> fine models re-centred on it -> equal-weight fine ensemble."""
import hashlib
import json
from pathlib import Path
import numpy as np
import torch
from stages import fine_decoders
from inference import load_fold_model, fine_infer

def coarse_ensemble(root, runs, key):
    oof = np.mean([np.load(Path(root) / r / 'oof.npz')[key] for r in runs], 0)
    test = np.mean([np.load(Path(root) / r / 'test.npz')[key] for r in runs], 0)
    return oof, test

def recentre_fine(root, run, train, test, oof_centers, test_centers, device, iters=2, center_dec='p25'):
    """Out-of-fold re-inference of one fine run around new centres; cached by a hash of the centres."""
    run_dir = Path(root) / run
    cfg = json.loads((run_dir / 'config.json').read_text())
    decoders = fine_decoders(cfg)
    digest = hashlib.sha256(np.round(np.r_[oof_centers.ravel(), test_centers.ravel()], 3).tobytes()
                            + json.dumps(decoders, sort_keys=True).encode()).hexdigest()[:12]
    cache = run_dir / f'recentre_{digest}_{center_dec}_it{iters}.npz'
    if cache.exists():
        z = np.load(cache)
        return {k[4:]: z[k] for k in z.files if k.startswith('oof|')}, {k[5:]: z[k] for k in z.files if k.startswith('test|')}
    centers = dict(zip([fd.name for fd in train], oof_centers))
    centers.update(zip([fd.name for fd in test], test_centers))
    idx = {fd.name: i for i, fd in enumerate(train)}
    oof, tests = {}, []
    for k in range(5):
        model, cfg = load_fold_model(run_dir, k, device)
        va = [fd for fd in train if fd.fold == k]
        res = fine_infer(model, va, centers, cfg, device, decoders, iters=iters, center_key=(center_dec, 'flip'))
        for key, v in res.items():
            oof.setdefault('|'.join(map(str, key)), np.zeros((len(train), 2)))[[idx[fd.name] for fd in va]] = v
        tests.append(fine_infer(model, test, centers, cfg, device, decoders, iters=iters, center_key=(center_dec, 'flip')))
        del model
        if device.type == 'mps':
            torch.mps.empty_cache()
    test_mean = {'|'.join(map(str, key)): np.mean([t[key] for t in tests], 0) for key in tests[0]}
    np.savez(cache, **{f'oof|{k}': v for k, v in oof.items()}, **{f'test|{k}': v for k, v in test_mean.items()})
    return oof, test_mean

def final_predictions(root, train, test, device, coarse_runs, coarse_key, fine_runs, fine_key, iters=2,
                      center_dec='p25'):
    co_oof, co_test = coarse_ensemble(root, coarse_runs, coarse_key)
    fine = {r: recentre_fine(root, r, train, test, co_oof, co_test, device, iters, center_dec) for r in fine_runs}
    oof = np.mean([fine[r][0][fine_key] for r in fine_runs], 0)
    tst = np.mean([fine[r][1][fine_key] for r in fine_runs], 0)
    return dict(coarse_oof=co_oof, coarse_test=co_test, oof=oof, test=tst, per_run=fine)


In [ ]:
%%writefile /kaggle/working/task3_source/recipes.py
"""Completed coarse/fine experiment recipes and the final fixed cascade members."""
RECIPES = {'c2_r34_256': {'aug': {'geo': True, 'photo': 1.0, 'rot': 20, 'scale': 1.12, 'shift': 0.05},
                'batch': 8,
                'coord_w': 1.0,
                'drop': 0.0,
                'ema': 0.995,
                'encoder': 'resnet34',
                'lr_dec': 0.001,
                'lr_enc': 0.0003,
                'save_weights': True,
                'sigma': 6.0,
                'size': 256,
                'steps': 1000,
                'val_every': 100,
                'wd': 0.01},
 'c_abl_cw0': {'aug': {'geo': True, 'photo': 1.0, 'rot': 20, 'scale': 1.12, 'shift': 0.05},
               'batch': 8,
               'coord_w': 0,
               'drop': 0.0,
               'ema': 0.995,
               'encoder': 'resnet34',
               'lr_dec': 0.001,
               'lr_enc': 0.0003,
               'sigma': 6.0,
               'size': 256,
               'steps': 1000,
               'val_every': 100,
               'wd': 0.01},
 'c_abl_imgcrop': {'aug': {'crop': 'image',
                           'geo': True,
                           'photo': 1.0,
                           'rot': 20,
                           'scale': 1.12,
                           'shift': 0.05},
                   'batch': 8,
                   'coord_w': 1.0,
                   'drop': 0.0,
                   'ema': 0.995,
                   'encoder': 'resnet34',
                   'lr_dec': 0.001,
                   'lr_enc': 0.0003,
                   'sigma': 6.0,
                   'size': 256,
                   'steps': 1000,
                   'val_every': 100,
                   'wd': 0.01},
 'c_abl_nogeo': {'aug': {'geo': False, 'photo': 1.0, 'rot': 20, 'scale': 1.12, 'shift': 0.05},
                 'batch': 8,
                 'coord_w': 1.0,
                 'drop': 0.0,
                 'ema': 0.995,
                 'encoder': 'resnet34',
                 'lr_dec': 0.001,
                 'lr_enc': 0.0003,
                 'sigma': 6.0,
                 'size': 256,
                 'steps': 1000,
                 'val_every': 100,
                 'wd': 0.01},
 'c_abl_nophoto': {'aug': {'geo': True, 'photo': 0, 'rot': 20, 'scale': 1.12, 'shift': 0.05},
                   'batch': 8,
                   'coord_w': 1.0,
                   'drop': 0.0,
                   'ema': 0.995,
                   'encoder': 'resnet34',
                   'lr_dec': 0.001,
                   'lr_enc': 0.0003,
                   'sigma': 6.0,
                   'size': 256,
                   'steps': 1000,
                   'val_every': 100,
                   'wd': 0.01},
 'c_abl_scratch': {'aug': {'geo': True, 'photo': 1.0, 'rot': 20, 'scale': 1.12, 'shift': 0.05},
                   'batch': 8,
                   'coord_w': 1.0,
                   'drop': 0.0,
                   'ema': 0.995,
                   'encoder': 'resnet34',
                   'lr_dec': 0.001,
                   'lr_enc': 0.0003,
                   'pretrained': False,
                   'sigma': 6.0,
                   'size': 256,
                   'steps': 1000,
                   'val_every': 100,
                   'wd': 0.01},
 'c_r18_256': {'aug': {'geo': True, 'photo': 1.0, 'rot': 20, 'scale': 1.12, 'shift': 0.05},
               'batch': 8,
               'coord_w': 1.0,
               'drop': 0.0,
               'ema': 0.995,
               'encoder': 'resnet18',
               'lr_dec': 0.001,
               'lr_enc': 0.0003,
               'save_weights': True,
               'sigma': 6.0,
               'size': 256,
               'steps': 1000,
               'val_every': 100,
               'wd': 0.01},
 'c_r18_256_seed2': {'aug': {'geo': True, 'photo': 1.0, 'rot': 20, 'scale': 1.12, 'shift': 0.05},
                     'batch': 8,
                     'coord_w': 1.0,
                     'drop': 0.0,
                     'ema': 0.995,
                     'encoder': 'resnet18',
                     'lr_dec': 0.001,
                     'lr_enc': 0.0003,
                     'save_weights': True,
                     'seed_offset': 100,
                     'sigma': 6.0,
                     'size': 256,
                     'steps': 1000,
                     'val_every': 100,
                     'wd': 0.01},
 'c_r34_256': {'aug': {'geo': True, 'photo': 1.0, 'rot': 20, 'scale': 1.12, 'shift': 0.05},
               'batch': 8,
               'coord_w': 1.0,
               'drop': 0.0,
               'ema': 0.995,
               'encoder': 'resnet34',
               'lr_dec': 0.001,
               'lr_enc': 0.0003,
               'save_weights': True,
               'sigma': 6.0,
               'size': 256,
               'steps': 1000,
               'val_every': 100,
               'wd': 0.01},
 'c_r34_256_fbn': {'aug': {'geo': True, 'photo': 1.0, 'rot': 20, 'scale': 1.12, 'shift': 0.05},
                   'batch': 8,
                   'coord_w': 1.0,
                   'drop': 0.0,
                   'ema': 0.995,
                   'encoder': 'resnet34',
                   'freeze_bn': True,
                   'lr_dec': 0.001,
                   'lr_enc': 0.0003,
                   'save_weights': True,
                   'sigma': 6.0,
                   'size': 256,
                   'steps': 1000,
                   'val_every': 100,
                   'wd': 0.01},
 'c_r34_256_reg': {'aug': {'erase': 0.5, 'geo': True, 'photo': 1.0, 'rot': 30, 'scale': 1.2, 'shift': 0.05},
                   'batch': 8,
                   'coord_w': 1.0,
                   'drop': 0.1,
                   'ema': 0.995,
                   'encoder': 'resnet34',
                   'lr_dec': 0.001,
                   'lr_enc': 0.0003,
                   'save_weights': True,
                   'sigma': 6.0,
                   'size': 256,
                   'steps': 1000,
                   'val_every': 100,
                   'wd': 0.01},
 'c_r34_256_s400': {'aug': {'geo': True, 'photo': 1.0, 'rot': 20, 'scale': 1.12, 'shift': 0.05},
                    'batch': 8,
                    'coord_w': 1.0,
                    'drop': 0.0,
                    'ema': 0.995,
                    'encoder': 'resnet34',
                    'lr_dec': 0.001,
                    'lr_enc': 0.0003,
                    'save_weights': True,
                    'sigma': 6.0,
                    'size': 256,
                    'steps': 400,
                    'val_every': 50,
                    'wd': 0.01},
 'c_r34_256_s400_reg': {'aug': {'erase': 0.5,
                                'geo': True,
                                'photo': 1.0,
                                'rot': 30,
                                'scale': 1.2,
                                'shift': 0.05},
                        'batch': 8,
                        'coord_w': 1.0,
                        'drop': 0.1,
                        'ema': 0.995,
                        'encoder': 'resnet34',
                        'lr_dec': 0.001,
                        'lr_enc': 0.0003,
                        'save_weights': True,
                        'sigma': 6.0,
                        'size': 256,
                        'steps': 400,
                        'val_every': 50,
                        'wd': 0.01},
 'c_r34_256_s8lr1': {'aug': {'geo': True, 'photo': 1.0, 'rot': 20, 'scale': 1.12, 'shift': 0.05},
                     'batch': 8,
                     'coord_w': 1.0,
                     'drop': 0.0,
                     'ema': 0.995,
                     'encoder': 'resnet34',
                     'lr_dec': 0.001,
                     'lr_enc': 0.0001,
                     'save_weights': True,
                     'sigma': 8,
                     'size': 256,
                     'steps': 1000,
                     'val_every': 100,
                     'wd': 0.01},
 'c_r34_256_seed2': {'aug': {'geo': True, 'photo': 1.0, 'rot': 20, 'scale': 1.12, 'shift': 0.05},
                     'batch': 8,
                     'coord_w': 1.0,
                     'drop': 0.0,
                     'ema': 0.995,
                     'encoder': 'resnet34',
                     'lr_dec': 0.001,
                     'lr_enc': 0.0003,
                     'save_weights': True,
                     'seed_offset': 100,
                     'sigma': 6.0,
                     'size': 256,
                     'steps': 1000,
                     'val_every': 100,
                     'wd': 0.01},
 'c_r34_384': {'aug': {'geo': True, 'photo': 1.0, 'rot': 20, 'scale': 1.12, 'shift': 0.05},
               'batch': 8,
               'coord_w': 1.0,
               'drop': 0.0,
               'ema': 0.995,
               'encoder': 'resnet34',
               'lr_dec': 0.001,
               'lr_enc': 0.0003,
               'save_weights': True,
               'sigma': 9,
               'size': 384,
               'steps': 1000,
               'val_every': 100,
               'wd': 0.01},
 'f2_640_wide': {'aug': {'geo': True, 'jitter': 0.06, 'photo': 1.0, 'rot': 25, 'scale': 1.12, 'wide': 0.28},
                 'batch': 8,
                 'coarse': 'c2_r34_256',
                 'coarse_key': 'global|flip',
                 'coord_w': 1.0,
                 'crop': 640,
                 'drop': 0.0,
                 'ema': 0.995,
                 'encoder': 'resnet34',
                 'iters': 2,
                 'lr_dec': 0.001,
                 'lr_enc': 0.0003,
                 'save_weights': True,
                 'sigma': 6.0,
                 'size': 256,
                 'steps': 600,
                 'val_every': 100,
                 'wd': 0.01},
 'f480_narrow_s600': {'aug': {'geo': True,
                              'jitter': 0.06,
                              'jitter_px': 25,
                              'photo': 1.0,
                              'rot': 25,
                              'scale': 1.12,
                              'wide': 0.28,
                              'wide_px': 80},
                      'batch': 8,
                      'coarse': 'c_r34_256',
                      'coarse_key': 'global|flip',
                      'coord_w': 1.0,
                      'crop': 480,
                      'drop': 0.0,
                      'ema': 0.995,
                      'encoder': 'resnet34',
                      'iters': 2,
                      'lr_dec': 0.001,
                      'lr_enc': 0.0003,
                      'save_weights': True,
                      'sigma': 6.0,
                      'size': 256,
                      'steps': 600,
                      'val_every': 100,
                      'wd': 0.01},
 'f480_wide_s600': {'aug': {'geo': True,
                            'jitter': 0.06,
                            'jitter_px': 38,
                            'photo': 1.0,
                            'rot': 25,
                            'scale': 1.12,
                            'wide': 0.28,
                            'wide_px': 179},
                    'batch': 8,
                    'coarse': 'c_r34_256',
                    'coarse_key': 'global|flip',
                    'coord_w': 1.0,
                    'crop': 480,
                    'drop': 0.0,
                    'ema': 0.995,
                    'encoder': 'resnet34',
                    'iters': 2,
                    'lr_dec': 0.001,
                    'lr_enc': 0.0003,
                    'save_weights': True,
                    'sigma': 6.0,
                    'size': 256,
                    'steps': 600,
                    'val_every': 100,
                    'wd': 0.01},
 'f640_narrow_s600': {'aug': {'geo': True,
                              'jitter': 0.06,
                              'jitter_px': 25,
                              'photo': 1.0,
                              'rot': 25,
                              'scale': 1.12,
                              'wide': 0.28,
                              'wide_px': 80},
                      'batch': 8,
                      'coarse': 'c_r34_256',
                      'coarse_key': 'global|flip',
                      'coord_w': 1.0,
                      'crop': 640,
                      'drop': 0.0,
                      'ema': 0.995,
                      'encoder': 'resnet34',
                      'iters': 2,
                      'lr_dec': 0.001,
                      'lr_enc': 0.0003,
                      'save_weights': True,
                      'sigma': 6.0,
                      'size': 256,
                      'steps': 600,
                      'val_every': 100,
                      'wd': 0.01},
 'f640_wide_384': {'aug': {'geo': True, 'jitter': 0.06, 'photo': 1.0, 'rot': 25, 'scale': 1.12, 'wide': 0.28},
                   'batch': 8,
                   'coarse': 'c_r34_256',
                   'coarse_key': 'global|flip',
                   'coord_w': 1.0,
                   'crop': 640,
                   'drop': 0.0,
                   'ema': 0.995,
                   'encoder': 'resnet34',
                   'iters': 2,
                   'lr_dec': 0.001,
                   'lr_enc': 0.0003,
                   'save_weights': True,
                   'sigma': 9,
                   'size': 384,
                   'steps': 600,
                   'val_every': 100,
                   'wd': 0.01},
 'f640_wide_s600': {'aug': {'geo': True,
                            'jitter': 0.06,
                            'photo': 1.0,
                            'rot': 25,
                            'scale': 1.12,
                            'wide': 0.28},
                    'batch': 8,
                    'coarse': 'c_r34_256',
                    'coarse_key': 'global|flip',
                    'coord_w': 1.0,
                    'crop': 640,
                    'drop': 0.0,
                    'ema': 0.995,
                    'encoder': 'resnet34',
                    'iters': 2,
                    'lr_dec': 0.001,
                    'lr_enc': 0.0003,
                    'save_weights': True,
                    'sigma': 6.0,
                    'size': 256,
                    'steps': 600,
                    'val_every': 100,
                    'wd': 0.01},
 'f640_wide_s600_seed2': {'aug': {'geo': True,
                                  'jitter': 0.06,
                                  'photo': 1.0,
                                  'rot': 25,
                                  'scale': 1.12,
                                  'wide': 0.28},
                          'batch': 8,
                          'coarse': 'c_r34_256',
                          'coarse_key': 'global|flip',
                          'coord_w': 1.0,
                          'crop': 640,
                          'drop': 0.0,
                          'ema': 0.995,
                          'encoder': 'resnet34',
                          'iters': 2,
                          'lr_dec': 0.001,
                          'lr_enc': 0.0003,
                          'save_weights': True,
                          'seed_offset': 100,
                          'sigma': 6.0,
                          'size': 256,
                          'steps': 600,
                          'val_every': 100,
                          'wd': 0.01},
 'f896_wide_s600': {'aug': {'geo': True,
                            'jitter': 0.06,
                            'jitter_px': 38,
                            'photo': 1.0,
                            'rot': 25,
                            'scale': 1.12,
                            'wide': 0.28,
                            'wide_px': 179},
                    'batch': 8,
                    'coarse': 'c_r34_256',
                    'coarse_key': 'global|flip',
                    'coord_w': 1.0,
                    'crop': 896,
                    'drop': 0.0,
                    'ema': 0.995,
                    'encoder': 'resnet34',
                    'iters': 2,
                    'lr_dec': 0.001,
                    'lr_enc': 0.0003,
                    'save_weights': True,
                    'sigma': 6.0,
                    'size': 256,
                    'steps': 600,
                    'val_every': 100,
                    'wd': 0.01},
 'f_r34_640': {'aug': {'geo': True, 'jitter': 0.06, 'photo': 1.0, 'rot': 25, 'scale': 1.12, 'wide': 0.28},
               'batch': 8,
               'coarse': 'c_r34_256',
               'coarse_key': 'global|flip',
               'coord_w': 1.0,
               'crop': 640,
               'drop': 0.0,
               'ema': 0.995,
               'encoder': 'resnet34',
               'iters': 2,
               'lr_dec': 0.001,
               'lr_enc': 0.0003,
               'save_weights': True,
               'sigma': 6.0,
               'size': 256,
               'steps': 1000,
               'val_every': 100,
               'wd': 0.01}}

FINAL_COARSE = ['c_r34_256', 'c_r34_256_seed2', 'c_r18_256', 'c_r18_256_seed2', 'c_r34_256_fbn', 'c_r34_256_reg']
FINAL_COARSE_KEY = 'global|flip'
FINAL_FINE = ['f640_wide_s600', 'f640_wide_s600_seed2', 'f896_wide_s600', 'f640_wide_384']
FINE_KEY = 'p15|flip|it1'

SECOND_SPLIT = {"c2_r34_256", "f2_640_wide"}


In [ ]:
%%writefile /kaggle/working/task3_source/train.py
"""Train the fixed coarse-to-fine ensemble or the completed controlled experiments."""
import argparse
import copy
import json
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from data import load_dataset, find_data_dir, SPLIT2_FOLDS, dataset_fingerprint
from stages import run_coarse, run_fine, write_submission
from cascade import final_predictions
from training import default_device
from recipes import RECIPES, SECOND_SPLIT, FINAL_COARSE, FINAL_COARSE_KEY, FINAL_FINE, FINE_KEY


def train_recipes(names, data, output, device, folds=range(5), config_overrides=None):
    first, test, sample = load_dataset(data)
    second = []
    for item in first:
        other = copy.copy(item)
        other.fold = int(SPLIT2_FOLDS[Path(item.name).stem])
        second.append(other)
    root = output / 'runs'
    for name in names:
        config = copy.deepcopy(RECIPES[name])
        config.update((config_overrides or {}).get(name, {}))
        items = second if name in SECOND_SPLIT else first
        if 'coarse' in config:
            source, key = config.pop('coarse'), config.pop('coarse_key')
            run_fine(name, config, source, key, items, test, root, str(device), folds)
        else:
            run_coarse(name, config, items, test, root, str(device), folds)
    return first, test, sample


def export_cascade(root, train, test, sample, device, data):
    result = final_predictions(root / 'runs', train, test, device, FINAL_COARSE, FINAL_COARSE_KEY,
                               FINAL_FINE, FINE_KEY, iters=1, center_dec='p25')
    np.savez_compressed(root / 'cascade_predictions.npz', oof=result['oof'], test=result['test'],
                        train_names=np.asarray([item.name for item in train]),
                        test_names=np.asarray([item.name for item in test]),
                        target=np.asarray([item.xy for item in train]),
                        folds=np.asarray([item.fold for item in train]),
                        groups=np.asarray([item.group for item in train]), dataset_fingerprint=dataset_fingerprint(data))
    write_submission(test, result['test'], sample, root / 'submission_cascade.csv')
    return result


def main(argv=None):
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument('--data', default=None)
    parser.add_argument('--out', type=Path, default=Path('outputs'))
    parser.add_argument('--group', choices=['final', 'experiments'], default='final')
    parser.add_argument('--run', choices=sorted(RECIPES))
    parser.add_argument('--folds', nargs='+', type=int, choices=range(5), default=list(range(5)))
    parser.add_argument('--device', default=str(default_device()))
    parser.add_argument('--list', action='store_true')
    args = parser.parse_args(argv)
    if args.list:
        print('\n'.join(RECIPES))
        return
    names = FINAL_COARSE + FINAL_FINE if args.group == 'final' else list(RECIPES)
    if args.run:
        names = [args.run]
        if 'coarse' in RECIPES[args.run]:
            names.insert(0, RECIPES[args.run]['coarse'])
    # Coarse outputs must exist before dependent fine runs.
    names = [name for name in names if 'coarse' not in RECIPES[name]] + [name for name in names if 'coarse' in RECIPES[name]]
    names = list(dict.fromkeys(names))
    args.out.mkdir(parents=True, exist_ok=True)
    train, test, sample = train_recipes(names, find_data_dir(args.data), args.out, torch.device(args.device), args.folds)
    ready = all((args.out / 'runs' / name / f'model_fold{fold}.pt').is_file()
                for name in FINAL_COARSE + FINAL_FINE for fold in range(5))
    if ready:
        export_cascade(args.out, train, test, sample, torch.device(args.device), find_data_dir(args.data))


if __name__ == '__main__':
    main()


In [ ]:
import subprocess
GROUP = 'final'
subprocess.run(["python", "/kaggle/working/task3_source/train.py", "--data", "/kaggle/input", "--out", "/kaggle/working/task3", "--group", GROUP], check=True)
